# SOLARNESIA: Spatial Model & GIS Analytics for National Solar Power Plant (PLTS) Distribution
**Mata Kuliah:** IMT01303501 - Komputer Grafis (3 SKS)  
**Kelompok 6:** Wesley Goeinawan (0706012410011), Stevanus Ivan Santoso (0706012410029), Nicholas Gerwin Mawardji (0706012410015)  
**Program Studi:** Informatika, Universitas Ciputra Surabaya (2026/2027)

---
### Ringkasan Alur Pemrosesan Spasial
1. **Layer 1 (2D MultiPolygons):** Batas daratan kepulauan dan batas administrasi kabupaten/provinsi (Choropleth rata-rata potensi PVOUT nasional).
2. **Layer 2 (0D Points & Buffers):** Titik observasi Global Solar Atlas darat murni (3.802 titik) setelah *Spatial Clip* mengeliminasi 1.160 titik laut.
3. **5 km Hotspot Clustering:** Pengelompokan titik kandidat berdekatan (≤ 5 km) menjadi sentra hotspot terintegrasi (DBSCAN).
4. **Operasi 1 (Buffer Metrik):** Proyeksi EPSG:3857 dengan koreksi $\cos(\text{lat})$, pembentukan *footprint* 3 km `buffer(3000)` dan koridor transmisi 25 km `buffer(25000)` (standar PLN 20 kV).
5. **Operasi 2 (Spatial Join & Overlay):** Menghitung keterjangkauan 3.289 permukiman darat serta mengekstrak daftar kecamatan/kabupaten yang tercakup secara interaktif.

In [1]:
# Cell 0: Setup & Library Imports
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch, Circle
from matplotlib.lines import Line2D
from shapely.geometry import Point, Polygon
from sklearn.cluster import DBSCAN
import folium

print(f"GeoPandas: {gpd.__version__} | Pandas: {pd.__version__} | Folium: {folium.__version__}")
os.makedirs("outputs", exist_ok=True)
print("Output directory ready.")


GeoPandas: 1.2.0 | Pandas: 3.0.6 | Folium: 0.20.0
Output directory ready.


In [ ]:
# Cell 1: Ingestion & GeoDataFrame Construction

In [ ]:
# Cell 2: Spatial Clip / Land-Masking (Op 0)

In [ ]:
# Cell 3: Site Selection & Role Classification (Layer 2)

In [ ]:
# Cell 4: 5 km Proximity Hotspot Clustering (DBSCAN)
# Mengelompokkan titik kandidat yang berjarak <= 5 km menjadi satu klaster sentra energi terpadu
plts_3857 = plts_candidates.to_crs("EPSG:3857")
coords_m = np.column_stack([plts_3857.geometry.x, plts_3857.geometry.y])

# Clustering dengan radius ketetanggaan 5.000 meter
db = DBSCAN(eps=5000, min_samples=1, metric='euclidean').fit(coords_m)
plts_candidates['hotspot_id'] = [f"HOTSPOT-{cid:03d}" for cid in db.labels_]
plts_3857['hotspot_id'] = plts_candidates['hotspot_id']

unique_hotspots = plts_candidates['hotspot_id'].nunique()
cluster_sizes = plts_candidates['hotspot_id'].value_counts()
multi_site_hotspots = (cluster_sizes > 1).sum()

print(f"Total Kandidat PLTS      : {len(plts_candidates)} titik")
print(f"Total Hotspot Sentra     : {unique_hotspots} klaster")
print(f"Hotspot Multi-titik (<=5km): {multi_site_hotspots} klaster terpadu")

assert unique_hotspots == 480, f"Expected 480 hotspots, got {unique_hotspots}"
assert multi_site_hotspots == 31, f"Expected 31 multi-site clusters, got {multi_site_hotspots}"
print("5km Proximity Clustering verified.")


In [ ]:
# Cell 5: Transformasi CRS & Pembuatan Buffer Metrik (Op 1)
# Transformasi ke EPSG:3857 dengan koreksi distorsi Mercator: R_adj = R / cos(latitude)
demand_3857 = demand_points.to_crs("EPSG:3857")

lats_plts = plts_candidates['latitude'].values
rad_adj_3k = 3000.0 / np.cos(np.radians(lats_plts))
rad_adj_25k = 25000.0 / np.cos(np.radians(lats_plts))

# Buffer 3 km (Footprint Lahan) dan Buffer 25 km (Standar Distribusi 20 kV PLN)
plts_3857['footprint_3km'] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_3k)]
plts_3857['buffer_25km'] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_25k)]

# Buffer Sensitivitas 10 km dan 40 km
rad_adj_10k = 10000.0 / np.cos(np.radians(lats_plts))
rad_adj_40k = 40000.0 / np.cos(np.radians(lats_plts))
plts_3857['buffer_10km'] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_10k)]
plts_3857['buffer_40km'] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_40k)]

# Poligon Union Koridor Distribusi
union_25km = gpd.GeoSeries(plts_3857['buffer_25km'], crs="EPSG:3857").union_all()
union_10km = gpd.GeoSeries(plts_3857['buffer_10km'], crs="EPSG:3857").union_all()
union_40km = gpd.GeoSeries(plts_3857['buffer_40km'], crs="EPSG:3857").union_all() 
#gatau ini apa

print("Buffer polygons generated with latitude-scaling compensation.")


In [ ]:
# Cell 6: Analisis Keterjangkauan Permukiman Spasial (Op 2)
# Titik demand diuji posisinya terhadap poligon buffer distribusi menggunakan predikat intersects / within
demand_3857['covered_25km'] = demand_3857.geometry.within(union_25km)
demand_3857['covered_10km'] = demand_3857.geometry.within(union_10km)
demand_3857['covered_40km'] = demand_3857.geometry.within(union_40km)

reached_25 = int(demand_3857['covered_25km'].sum())
unreached_25 = len(demand_3857) - reached_25
pct_25 = (reached_25 / len(demand_3857)) * 100

reached_10 = int(demand_3857['covered_10km'].sum())
pct_10 = (reached_10 / len(demand_3857)) * 100

reached_40 = int(demand_3857['covered_40km'].sum())
pct_40 = (reached_40 / len(demand_3857)) * 100

print(f"Jangkauan 10 km (Sensitivitas)   : {reached_10} titik ({pct_10:.1f}%)")
print(f"Jangkauan 25 km (Standar PLN 20kV): {reached_25} titik ({pct_25:.1f}%)")
print(f"Jangkauan 40 km (Sensitivitas)   : {reached_40} titik ({pct_40:.1f}%)")
print(f"Permukiman Belum Terjangkau (>25km): {unreached_25} titik ({(unreached_25/len(demand_3857))*100:.1f}%)")

assert reached_25 == 325, f"Expected 325, got {reached_25}"
assert unreached_25 == 2964, f"Expected 2964, got {unreached_25}"
assert round(pct_25, 1) == 9.9
assert reached_10 == 49
assert reached_40 == 605
print("Demand reach statistics 100% match proposal.")


In [ ]:
# Cell 7: Ekstraksi Cakupan Wilayah Administratif per Titik / Hotspot
# Mengiriskan buffer 25 km tiap kandidat dengan poligon batas kabupaten/kota
kab_3857 = gdf_kab.to_crs("EPSG:3857")

covered_areas_list = []
print("Pre-calculating 25km administrative reach for each hotspot...")

for idx, row in plts_3857.iterrows():
    buf_geom = row['buffer_25km']
    intersected_kabs = kab_3857[kab_3857.geometry.intersects(buf_geom)]['shapeName'].unique()
    covered_str = ", ".join(sorted(intersected_kabs))
    covered_areas_list.append(covered_str)

plts_candidates['covered_areas_25km'] = covered_areas_list
plts_3857['covered_areas_25km'] = covered_areas_list


#pelajari

# Cakupan Nasional
all_covered_kabs = kab_3857[kab_3857.geometry.intersects(union_25km)]['shapeName'].unique()
print(f"Total Kabupaten/Kota Terjangkau Cincin PLTS: {len(all_covered_kabs)} dari {len(gdf_kab)} nasional")

# Uji Cek Titik Bali Selatan (Denpasar & Badung)
bali_pt = plts_candidates[(plts_candidates['latitude'] > -8.8) & (plts_candidates['latitude'] < -8.5) & (plts_candidates['longitude'] > 115.0) & (plts_candidates['longitude'] < 115.3)].iloc[0]
print("\n[Contoh Klik Titik Bali Selatan] ({:.3f}° BT, {:.3f}° LS):".format(bali_pt.longitude, bali_pt.latitude))
print(f"  Hotspot ID     : {bali_pt.hotspot_id}")
print(f"  Potensi Radiasi: {bali_pt.pvout:.1f} kWh/kWp | Suhu: {bali_pt.temp:.1f}°C")
print(f"  Wilayah 25 km  : {bali_pt.covered_areas_25km}")

assert len(all_covered_kabs) == 294, f"Expected 294 covered kabupaten, got {len(all_covered_kabs)}"
assert "Kota Denpasar" in bali_pt.covered_areas_25km and "Badung" in bali_pt.covered_areas_25km
print("\nAdministrative coverage assertions passed.")


In [ ]:
# Cell 8: Rangkuman Metrik & Ekspor Tabel 1

In [ ]:
# Cell 9: Visualisasi Komputer Grafis 2D Statis (Matplotlib)

In [ ]:
# Cell 10: Peta Spasial Interaktif & Inspeksi Klik (Folium)